# Chest X-Ray Multi-Modal System - Colab Runner

DSAI 413 Assignment 2. Runs end-to-end on a **T4 GPU** in Colab.

Before running anything below:
- `Runtime > Change runtime type > Hardware accelerator = T4 GPU`
- In Colab Secrets (key icon in left sidebar), set:
  - `HF_TOKEN` (Hugging Face read token for MedGemma + ColPali)
  - `KAGGLE_USERNAME` and `KAGGLE_KEY` (only needed if downloading from Kaggle; skip if using Drive copy)
- Mount your Drive so the cached MIMIC-CXR copy at `/content/drive/MyDrive/dsai413-data` is reused (no 16 GB re-download).

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DATA = "/content/drive/MyDrive/dsai413-data"
import os
assert os.path.isdir(DRIVE_DATA), f"Drive path missing: {DRIVE_DATA}"
print("Drive dataset root:", DRIVE_DATA)
!ls -la {DRIVE_DATA} | head -20

## 2. Clone the repo

In [ ]:
import os
REPO_URL = "https://github.com/ahmed-abdelfatah1/assignment-2.git"
REPO_DIR = "/content/assignment-2"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}

## 3. Install dependencies

`colpali-engine` from GitHub source (PyPI lags transformers-5.x fixes).

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -U "git+https://github.com/illuin-tech/colpali"

## 4. Authentication & env

HF token (required) and Kaggle creds (optional - only used if `--dataset-path` is NOT set in cell 6).

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
try:
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("Kaggle:", os.environ["KAGGLE_USERNAME"])
except Exception:
    print("Kaggle creds not set (OK if using Drive copy).")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

## 5. Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No CUDA - switch to T4 GPU runtime first."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 6. Build the 400-sample manifests from the Drive copy

`--dataset-path` points at the pre-extracted Drive folder, so kagglehub is bypassed entirely (no 16 GB download). Takes ~2 min.

In [ ]:
!python -u data/download.py --dataset-path {DRIVE_DATA}

## 7. (Optional) Build the QA dataset

Skip if `data/sample/qa_dataset.json` already exists in the repo.

In [ ]:
import os
if not os.path.exists("data/sample/qa_dataset.json"):
    !python -u data/build_qa_dataset.py
else:
    print("qa_dataset.json already exists, skipping.")

## 8. (Optional) Run evaluation

Produces `results/comparison.json` + `results/comparison.md`. Full run ~50 min on T4.

In [ ]:
# Quick smoke (~5 min):
# !python -u -m src.eval --limit-report 5 --limit-qa 5

# Full run (~50 min):
!python -u -m src.eval

In [ ]:
!cat results/comparison.md

## 9. (Optional) Cache built indexes back to Drive

Saves the slow-to-build indexes to Drive so next session skips Phase 4's 12-min ColPali embed.

In [ ]:
DRIVE_CACHE = "/content/drive/MyDrive/dsai413-cache"
import os
os.makedirs(DRIVE_CACHE, exist_ok=True)
!cp -r data/sample/colpali_index {DRIVE_CACHE}/ 2>/dev/null || true
!cp -f data/sample/clip_index.faiss {DRIVE_CACHE}/ 2>/dev/null || true
!cp -f data/sample/clip_index.manifest.csv {DRIVE_CACHE}/ 2>/dev/null || true
!cp -f data/sample/text_index.faiss {DRIVE_CACHE}/ 2>/dev/null || true
!cp -f data/sample/text_index.manifest.csv {DRIVE_CACHE}/ 2>/dev/null || true
!cp -f data/sample/qa_dataset.json {DRIVE_CACHE}/ 2>/dev/null || true
!ls -la {DRIVE_CACHE}

## 10. (Optional) Restore cached indexes from Drive on a fresh session

Run this after cell 6 (manifests) but before cell 8 (eval) on a fresh runtime, to skip index rebuilding.

In [ ]:
DRIVE_CACHE = "/content/drive/MyDrive/dsai413-cache"
!cp -r {DRIVE_CACHE}/colpali_index data/sample/ 2>/dev/null || true
!cp -f {DRIVE_CACHE}/clip_index.faiss data/sample/ 2>/dev/null || true
!cp -f {DRIVE_CACHE}/clip_index.manifest.csv data/sample/ 2>/dev/null || true
!cp -f {DRIVE_CACHE}/text_index.faiss data/sample/ 2>/dev/null || true
!cp -f {DRIVE_CACHE}/text_index.manifest.csv data/sample/ 2>/dev/null || true
!cp -f {DRIVE_CACHE}/qa_dataset.json data/sample/ 2>/dev/null || true
!ls -la data/sample/

## 11. Launch the Gradio demo

Prints a public `gradio.live` URL (share=True). First click per model loads weights (~60 s MedGemma, ~15 s ColPali). Keep cell running to keep the demo alive.

In [ ]:
!python -u app/app.py